# `parse_pdf` — extraction d'un PDF en DataFrames

Module testé : [`src/docpipeline/parsing/pdf/parse_pdf.py`](../src/docpipeline/parsing/pdf/parse_pdf.py).

Ouvre fitz une seule fois et retourne 4 sorties prêtes à être consommées en aval :

| Sortie | Schéma |
|---|---|
| `line_df`     | 1 ligne = 1 ligne de texte (`span_id`, `text`, bbox, font_name, font_size, bold, italic, color, is_invisible) |
| `image_df`    | 1 ligne = 1 image (bbox, dimensions, ratio de couverture) |
| `page_df`     | 1 ligne = 1 page (`page_type`, flags additifs, `extraction_strategy`, `tool`) |
| `doc_summary` | JSON synthèse : `source_tool`, `content_type`, `recommended_strategy`, `pages_needing_ocr` |

**page_type** (mutuellement exclusif) : `native`, `native_with_image`, `scanned`, `scanned_ocr_good`, `scanned_ocr_bad`, `mixed`, `empty`, `unknown`.
**extraction_strategy** : `native` (fitz direct), `ocr` (OCR obligatoire), `hybrid` (texte natif + OCR sur images), `skip` (page vide).

Démo ci-dessous : extraction sur un contrat MRH du corpus client (`data/CG contrats MRH/CG Assistance aux personnes AXA.pdf`). Pour le bench complet, voir [`05_bench_parse_pdf_js.ipynb`](05_bench_parse_pdf_js.ipynb).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

PDF = Path('../data/CG contrats MRH/CG Assistance aux personnes AXA.pdf')

pd.set_option('display.max_colwidth', 120)

result = parse_pdf(PDF)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()])
display(summary_df)

display(Markdown(f"### 2. `page_df` — {len(result['page_df'])} pages classifiées"))
display(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images', 'image_coverage_ratio']])

display(Markdown(f"### 3. `line_df` — {len(result['line_df'])} lignes (10 premières)"))
display(result['line_df'].head(10)[['span_id', 'page_num', 'text', 'font_name', 'font_size', 'bold', 'italic', 'color']])

display(Markdown(f"### 4. `image_df` — {len(result['image_df'])} image(s) embarquée(s)"))
display(result['image_df'])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


### 1. `doc_summary` — synthèse au niveau document

,key,value
0,pdf_hash,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014
1,file_size_bytes,239954
2,n_pages,20
3,source_category,design_tool
4,source_tool,Adobe InDesign
5,source_confidence,0.95
6,source_signals,"['creator~adobe indesign', 'xmp_history~adobe indesign']"
7,creator_raw,Adobe InDesign 14.0 (Macintosh)
8,producer_raw,Adobe PDF Library 15.0
9,pdf_version,1.6


### 2. `page_df` — 20 pages classifiées

,page_num,page_type,tool,extraction_strategy,char_count,n_images,image_coverage_ratio
0,0,unknown,Adobe InDesign,hybrid,68,0,0.0000
1,1,native,Adobe InDesign,native,1206,0,0.0000
2,2,native,Adobe InDesign,native,877,0,0.0000
3,3,native,Adobe InDesign,native,861,0,0.0000
4,4,native,Adobe InDesign,native,2787,0,0.0000
5,5,native,Adobe InDesign,native,3329,0,0.0000
6,6,native,Adobe InDesign,native,2983,0,0.0000
7,7,native,Adobe InDesign,native,3192,0,0.0000
8,8,native,Adobe InDesign,native,2109,0,0.0000
9,9,native,Adobe InDesign,native,2299,0,0.0000


### 3. `line_df` — 562 lignes (10 premières)

,span_id,page_num,text,font_name,font_size,bold,italic,color
0,p_0_0,0,Janvier 2021,SourceSansPro-Regular,8.500,False,False,
1,p_0_1,0,Conditions,PublicoHeadline-Bold,39.000,True,False,
2,p_0_2,0,générales,PublicoHeadline-Bold,39.000,True,False,
3,p_0_3,0,Assistance,PublicoHeadline-Bold,39.000,True,False,
4,p_0_4,0,aux personnes,PublicoHeadline-Bold,39.000,True,False,
5,p_0_5,0,Assistance,SourceSansPro-Bold,12.726,True,False,
6,p_1_6,1,"Pour bénéficier de l’ensemble des garanties ci-après énumérées, il est",SourceSansPro-Bold,10.000,True,False,#2E3092
7,p_1_7,1,"impératif de contacter, préalablement à toute intervention AXA Assistance",SourceSansPro-Bold,10.000,True,False,#2E3092
8,p_1_8,1,lors de l’incident :,SourceSansPro-Bold,10.000,True,False,#2E3092
9,p_1_9,1,n par téléphone au 01 55 92 26 92 ; ou,Wingdings-Regular,7.500,False,False,#2E3092


### 4. `image_df` — 12 image(s) embarquée(s)

,pdf_hash,page_num,image_num,bbox,width,height,coverage_ratio
0,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,0,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
1,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,1,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
2,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,2,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
3,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,3,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
4,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,4,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
5,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,5,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
6,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,6,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
7,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,7,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
8,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,8,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
9,4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014,19,9,"(221.84527587890625, 374.83673095703125, 234.2013397216797, 392.04254150390625)",52,72,0.000851
